# MedNorm-VI S2 Assertion Training

Status: IMPLEMENTED, PREFLIGHT_EXECUTED, READY_FOR_COLAB_SMOKE, NOT_TRAINED. Full training is disabled by default and requires `I_AUTHORIZE_S2_FULL_TRAINING`. The governed corpus currently has no trusted assertion supervision, so this notebook stops before model acquisition unless that measured fact changes. `internal_test` is prohibited.


Required operator sections: Colab, GPU, Drive, Pinned dependencies, Git commit, split hash, revision, Smoke mode, Full mode OFF, Resume, Checkpoint manifest, Return-to-repository.


In [ ]:
from pathlib import Path
import json
import os
import subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/MedNorm-VI')
REPO_DIR = Path('/content/MedNorm-VI')
CORPUS_DIR = REPO_DIR / 'data' / 'derived' / 'training_corpora' / 'mednorm_vi_training_v1'
SMOKE_OUTPUT_DIR = DRIVE_ROOT / 'artifacts' / 's2_assertion_smoke_v1'
FULL_OUTPUT_DIR = DRIVE_ROOT / 'artifacts' / 's2_assertion_full_v1'
MODEL_CACHE_DIR = DRIVE_ROOT / 'model_cache' / 'huggingface'

RUN_SMOKE_TRAINING = False
RUN_FULL_TRAINING = False
CONFIRM_FULL = ''
S2_FULL_AUTHORIZATION = 'I_AUTHORIZE_S2_FULL_TRAINING'
SEED = 20260728

if RUN_FULL_TRAINING and CONFIRM_FULL != S2_FULL_AUTHORIZATION:
    raise SystemExit('Full S2 training requires I_AUTHORIZE_S2_FULL_TRAINING')
if RUN_FULL_TRAINING and RUN_SMOKE_TRAINING:
    raise SystemExit('Choose smoke or full, not both')

OUTPUT_DIR = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else SMOKE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)
print('s2_output_dir', OUTPUT_DIR)


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('not running inside Colab')

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('repo_commit', commit)


In [ ]:
from mednorm_vi.training.phase2.s2_assertion_training import (
    S2_MODEL_ID,
    assert_no_document_leakage,
    assert_trainable,
    scan_assertion_supervision,
)

coverage = json.loads((CORPUS_DIR / 'manifests' / 'annotation_coverage.json').read_text())
train_report, train_examples = scan_assertion_supervision(
    'train', CORPUS_DIR / 'splits' / 'train.jsonl', coverage_manifest=coverage)
validation_report, validation_examples = scan_assertion_supervision(
    'validation', CORPUS_DIR / 'splits' / 'validation.jsonl', coverage_manifest=coverage)
assert_no_document_leakage(train_examples, validation_examples)

print(json.dumps({'train': train_report.as_dict(), 'validation': validation_report.as_dict()}, indent=2, sort_keys=True))
if not train_report.trainable:
    raise SystemExit('DATA_BLOCKED: governed corpus has no trusted S2 assertion supervision; labels are not fabricated')
assert_trainable(train_report)


In [ ]:
from mednorm_vi.training.phase2.s2_assertion_training import (
    ASSERTION_LABEL_ORDER,
    build_s2_assertion_head,
    build_s2_resolved_config,
    s2_config_sha256,
    s2_head_parameter_count,
)

MODEL_REVISION = os.environ.get('MEDNORM_S2_MODEL_REVISION', '')
TOKENIZER_REVISION = os.environ.get('MEDNORM_S2_TOKENIZER_REVISION', MODEL_REVISION)
if RUN_FULL_TRAINING and (len(MODEL_REVISION) != 40 or len(TOKENIZER_REVISION) != 40):
    raise SystemExit('Pinned 40-hex model/tokenizer revisions are required for full S2')

head = build_s2_assertion_head()
head_parameters = sum(p.numel() for p in head.parameters())
assert head_parameters == s2_head_parameter_count() == 9228

resolved_config = build_s2_resolved_config(
    mode='full' if RUN_FULL_TRAINING else 'smoke',
    model_id=S2_MODEL_ID,
    model_revision=MODEL_REVISION or '0' * 40,
    tokenizer_revision=TOKENIZER_REVISION or '0' * 40,
    seed=SEED,
    max_length=192,
    micro_batch_size=8,
    accumulation_steps=4,
    epochs=6 if RUN_FULL_TRAINING else 1,
    learning_rate=2e-5,
    progress={'heartbeat_first_n': 10, 'heartbeat_every_n_train': 100, 'heartbeat_every_n_validation': 50},
)
resolved_config['config_sha256'] = s2_config_sha256(resolved_config)
(OUTPUT_DIR / 'resolved_config.json').write_text(json.dumps(resolved_config, indent=2, sort_keys=True) + '\n')
print('label_order', ASSERTION_LABEL_ORDER, 'head_parameters', head_parameters)


In [ ]:
# Real training loop is intentionally not entered by default. It must use only
# supervised positions in train_examples; UNKNOWN labels are masked, not false.
if not (RUN_SMOKE_TRAINING or RUN_FULL_TRAINING):
    raise SystemExit('Set RUN_SMOKE_TRAINING=True after S2 supervision exists')


In [ ]:
from mednorm_vi.training.phase2.s2_assertion_training import validate_s2_artifact

report = validate_s2_artifact(OUTPUT_DIR, mode='full' if RUN_FULL_TRAINING else 'smoke')
print(json.dumps(report.as_dict(), indent=2, sort_keys=True))
if RUN_SMOKE_TRAINING or RUN_FULL_TRAINING:
    if not report.ok:
        raise AssertionError(report.failures)
